# Virginia Piedmont native seasonal patterns

This notebook analyzes seasonal patterns of native iNaturalist observations in the Virginia Piedmont using iNaturalist aggregate histogram endpoints.

The workflow is adaptive by taxon:
- `phenology` framing is used for phenology-relevant taxa (Butterflies and Moths, Flowering Plants).
- `seasonality` framing is used for other taxa (Mammals, Birds, Insects).
- Mixed selections in compare mode are labeled with neutral seasonal wording.

**Approach**
- `/observations/histogram` (`week_of_year`, `month_of_year`, and `year`) provides complete aggregate counts in a small number of calls.
- `native=true` delegates nativity filtering to iNaturalist's establishment-means logic.
- **Within-taxon normalization**: each taxon's monthly counts are expressed as a percentage of that taxon's annual total (bins sum to 100%), replacing effort-based normalization.
- Top-taxa detail uses `species_counts` plus monthly histograms for retained taxa.

**Tiered data loading (API load minimization)**
- **Tier 1 – Overview**: One histogram per top-level group (Insects, Birds, Mammals, Plants in Flower) using parent taxon IDs. ~8 API calls.
- **Tier 2 – Subgroup drill-down**: Expand one selected group into children (e.g. Insects → 9 orders). Fetched on demand.
- **Tier 3 – Species detail**: Top-N species for one selected subgroup only. Fetched on demand.
- All tiers cached independently; repeat runs cost zero API calls.

**Taxon-driven behavior**
- Taxon options are provided via `SEASONAL_TAXON_REGISTRY` from `helpers.py`.
- Hierarchical groups: Insects (9 subgroups), Birds (2 subgroups), Mammalia (standalone), Plants in Flower (standalone).
- Lepidoptera keeps phenotype life-stage data (adult, larva, pupa).
- "Other Insects" uses `without_taxon_id` to exclude named orders from the Insecta parent.
- "Non-passerines" uses `without_taxon_id` to exclude Passeriformes from Aves.

**Outputs**
- Weekly chart: raw counts plus within-taxon normalized share, with adaptive phenology/seasonality titles.
- Monthly chart(s): within-taxon prevalence for retained top taxa.
- Optional compare chart when compare mode is enabled.
- Phenology-specific life-stage coverage only when relevant.

In [1]:
from pathlib import Path
import sys
import importlib

import datetime as dt
import pandas as pd
import plotly.express as px
from IPython.display import display
from ipywidgets import Dropdown, SelectMultiple

repo_root = Path.cwd()
if not (repo_root / 'helpers.py').exists() and (repo_root.parent.parent / 'helpers.py').exists():
    repo_root = repo_root.parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import helpers as _helpers
_helpers = importlib.reload(_helpers)

from helpers import (
    get_inat_session,
    PIEDMONT_PLACE_IDS,
    SEASONAL_TAXON_REGISTRY,
    get_seasonal_taxon_by_key,
    get_seasonal_group_overview_entries,
    get_seasonal_subgroup_entries,
    fetch_seasonal_histograms,
    normalize_within_taxon,
    fetch_top_seasonal_taxa,
)

TOP_N = 25
MIN_OBS_PER_YEAR = 25
MAX_COMPARE_TAXA = 3
USE_COMPARE_MODE = False
REFRESH_CACHE = False
TIER3_RANK = 'species'  # Set to 'genus' (or another iNat rank) for higher-level Tier 3 taxa

CACHE_PATH = repo_root / 'notebooks' / 'exploratory' / 'cache' / 'va_piedmont_phenology'
CACHE_MAX_AGE_DAYS = 7
CACHE_TAG = 'v5_seasonal'

# -- Widgets --
# Primary: select top-level group
overview_entries = get_seasonal_group_overview_entries()
group_selector = Dropdown(
    options=[(e['title_label'], e['key']) for e in overview_entries],
    value='insects',
    description='Group:',
)

# Secondary: subgroup within the selected group (populated dynamically)
def _subgroup_options(group_key):
    subs = get_seasonal_subgroup_entries(group_key)
    return [(e['title_label'], e['key']) for e in subs]

subgroup_selector = SelectMultiple(
    options=_subgroup_options(group_selector.value),
    description='Subgroups:',
)

def _on_group_change(change):
    subgroup_selector.options = _subgroup_options(change['new'])
    subgroup_selector.value = ()

group_selector.observe(_on_group_change, names='value')

display(group_selector)
display(subgroup_selector)


def apply_plot_style(fig, title_text: str, height: int = 700):
    fig.update_layout(
        title={'text': title_text, 'x': 0.5, 'xanchor': 'center'},
        template='plotly_white',
        height=height,
        hovermode='x unified',
        legend={
            'orientation': 'h',
            'yanchor': 'bottom',
            'y': 1.02,
            'xanchor': 'left',
            'x': 0.0,
            'bgcolor': 'rgba(255,255,255,0.85)',
        },
        margin={'l': 70, 'r': 35, 'b': 60, 't': 85},
        font={'size': 12},
        paper_bgcolor='white',
        plot_bgcolor='rgba(246,248,251,0.65)',
    )
    fig.update_xaxes(showgrid=True, gridcolor='rgba(200,200,200,0.25)', zeroline=False)
    fig.update_yaxes(showgrid=True, gridcolor='rgba(200,200,200,0.25)', zeroline=False)
    return fig


session = get_inat_session()

print('Seasonal taxon groups available:')
display(pd.DataFrame([
    {
        'key': t['key'],
        'label': t['title_label'],
        'taxon_id': t['taxon_id'],
        'group': t['group'] or '(top-level)',
        'mode': t['analysis_mode'],
        'children': len(t.get('children', [])),
    }
    for t in SEASONAL_TAXON_REGISTRY
]))

print(f'\nRefresh cache: {REFRESH_CACHE} | Cache TTL: {CACHE_MAX_AGE_DAYS} days | Tier 3 rank: {TIER3_RANK}')

print('\nRun on: '+dt.datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

Dropdown(description='Group:', options=(('Insects', 'insects'), ('Birds', 'birds'), ('Mammals', 'mammalia'), (…

SelectMultiple(description='Subgroups:', options=(('Beetles', 'coleoptera'), ('Flies', 'diptera'), ('Ants, Bee…

Seasonal taxon groups available:


,key,label,taxon_id,group,mode,children
0,insects,Insects,47158,(top-level),seasonality,9
1,coleoptera,Beetles,47208,insects,seasonality,0
2,diptera,Flies,47822,insects,seasonality,0
3,hymenoptera,"Ants, Bees, and Wasps",47201,insects,seasonality,0
4,lepidoptera,Butterflies and Moths,47157,insects,phenology,0
5,hemiptera,True Bugs,47744,insects,seasonality,0
6,orthoptera,Grasshoppers and Crickets,47651,insects,seasonality,0
7,odonata,Dragonflies and Damselflies,47792,insects,seasonality,0
8,blattodea,Cockroaches and Termites,81769,insects,seasonality,0
9,other_insects,Other Insects,47158,insects,seasonality,0



Refresh cache: False | Cache TTL: 7 days | Tier 3 rank: species

Run on: 2026-04-04 15:40:28


In [2]:
# -- Tier 1: Overview histograms for all 4 top-level groups --
# 4 groups x 2 intervals = 8 API calls (cached after first run)

overview_entries = get_seasonal_group_overview_entries()

overview_hist = fetch_seasonal_histograms(
    overview_entries,
    PIEDMONT_PLACE_IDS,
    session=session,
    intervals=('week_of_year', 'month_of_year'),
    native=True,
    cache_path=CACHE_PATH,
    cache_tag=CACHE_TAG,
    cache_max_age_days=CACHE_MAX_AGE_DAYS,
    refresh_cache=REFRESH_CACHE,
)
overview_hist = normalize_within_taxon(overview_hist)

print('Tier 1 overview — weekly native histogram sums by group:')
for entry in overview_entries:
    fg = entry['focus_group']
    total = overview_hist[
        (overview_hist['focus_group'] == fg)
        & (overview_hist['life_stage_bucket'] == 'overall')
        & (overview_hist['interval'] == 'week_of_year')
    ]['count'].sum()
    print(f"  {entry['title_label']}: {int(total):,}")


## This overview is maybe unnecessary


# -- Tier 1 Weekly overview: all groups, within-taxon normalized --
import plotly.graph_objects as go
from plotly.subplots import make_subplots

overview_entries = get_seasonal_group_overview_entries()
focus_to_entry = {e['focus_group']: e for e in overview_entries}

weeks = list(range(1, 53))
month_ticks = [1, 5, 9, 14, 18, 22, 27, 31, 35, 40, 44, 48]
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

weekly = overview_hist[
    (overview_hist['interval'] == 'week_of_year')
    & (overview_hist['life_stage_bucket'] == 'overall')
].copy()

if weekly.empty:
    print('No weekly data found.')
else:
    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        subplot_titles=(
            'Raw native observations by week (all groups)',
            '% of annual observations by week (within-taxon normalized)',
        ),
        vertical_spacing=0.09,
    )

    for focus_group, group_df in weekly.groupby('focus_group'):
        entry = focus_to_entry.get(focus_group)
        if entry is None:
            continue
        group_df = group_df.set_index('bin').reindex(weeks, fill_value=0)
        line = {'color': entry['color'], 'width': 2.2}
        name = entry['title_label']

        fig.add_scatter(x=weeks, y=group_df['count'], mode='lines',
                        name=name, line=line, legendgroup=name, row=1, col=1)
        fig.add_scatter(x=weeks, y=group_df['fraction'], mode='lines',
                        name=name, line=line, legendgroup=name, showlegend=False, row=2, col=1)

    fig = apply_plot_style(fig, title_text='Weekly native seasonal overview (VA Piedmont)', height=760)
    fig.update_xaxes(tickvals=month_ticks, ticktext=month_names)
    fig.update_yaxes(title_text='Observations', row=1, col=1)
    fig.update_yaxes(title_text='% of annual observations', tickformat='.1%', row=2, col=1)
    fig.show()

print('\nRun on: '+dt.datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

Tier 1 overview — weekly native histogram sums by group:
  Insects: 353,485
  Birds: 174,065
  Mammals: 38,521
  Flowering Plants: 28,329



Run on: 2026-04-04 15:40:39


In [3]:
# -- Tier 2: Subgroup drill-down for selected group --
# Fetches histograms for all children of the selected group.
# Insects: 9 subgroups x 2 intervals = 18 calls (+ Lep life stages = 24)
# Birds: 2 subgroups x 2 intervals = 4 calls

selected_group_key = group_selector.value
subgroup_entries = get_seasonal_subgroup_entries(selected_group_key)
parent_entry = get_seasonal_taxon_by_key(selected_group_key)

subgroup_hist = fetch_seasonal_histograms(
    subgroup_entries,
    PIEDMONT_PLACE_IDS,
    session=session,
    intervals=('week_of_year', 'month_of_year'),
    native=True,
    cache_path=CACHE_PATH,
    cache_tag=CACHE_TAG,
    cache_max_age_days=CACHE_MAX_AGE_DAYS,
    refresh_cache=REFRESH_CACHE,
)
subgroup_hist = normalize_within_taxon(subgroup_hist)

print(f"Tier 2 subgroup drill-down for {parent_entry['title_label']}:")
for entry in subgroup_entries:
    fg = entry['focus_group']
    total = subgroup_hist[
        (subgroup_hist['focus_group'] == fg)
        & (subgroup_hist['life_stage_bucket'] == 'overall')
        & (subgroup_hist['interval'] == 'week_of_year')
    ]['count'].sum()
    print(f"  {entry['title_label']}: {int(total):,}")


## we probably only need to do this if there are subgroups on interest... which perhaps their should be?


# -- Tier 2 chart: subgroup weekly comparison (within-taxon normalized) --
import plotly.graph_objects as go
from plotly.subplots import make_subplots

focus_to_sub = {e['focus_group']: e for e in subgroup_entries}
life_stage_dash = {'overall': 'solid', 'adult': 'dash', 'larva': 'dot', 'pupa': 'dashdot'}

weekly_sub = subgroup_hist[
    (subgroup_hist['interval'] == 'week_of_year')
].copy()

if weekly_sub.empty:
    print('No subgroup weekly data found.')
else:
    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        subplot_titles=(
            f"Subgroup raw counts by week — {parent_entry['title_label']}",
            f"% of annual observations by week — {parent_entry['title_label']}",
        ),
        vertical_spacing=0.09,
    )

    for (focus_group, life_stage), gdf in weekly_sub.groupby(['focus_group', 'life_stage_bucket']):
        entry = focus_to_sub.get(focus_group)
        if entry is None:
            continue
        gdf = gdf.set_index('bin').reindex(weeks, fill_value=0)
        line = {'color': entry['color'], 'dash': life_stage_dash.get(life_stage, 'solid'), 'width': 2.0}
        name = f"{entry['title_label']} - {life_stage}"

        fig.add_scatter(x=weeks, y=gdf['count'], mode='lines',
                        name=name, line=line, legendgroup=name, row=1, col=1)
        fig.add_scatter(x=weeks, y=gdf['fraction'], mode='lines',
                        name=name, line=line, legendgroup=name, showlegend=False, row=2, col=1)

    fig = apply_plot_style(
        fig,
        title_text=f"Subgroup seasonal comparison — {parent_entry['title_label']} (VA Piedmont)",
        height=760,
    )
    fig.update_xaxes(tickvals=month_ticks, ticktext=month_names)
    fig.update_yaxes(title_text='Observations', row=1, col=1)
    fig.update_yaxes(title_text='% of annual observations', tickformat='.1%', row=2, col=1)
    fig.show()

print('\nRun on: '+dt.datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

Tier 2 subgroup drill-down for Mammals:
  Mammals: 38,521



Run on: 2026-04-04 15:40:43


In [6]:
# -- Tier 3: Top species for selected subgroup --
# Select a single subgroup to drill into.
# Uses species_counts + year histogram + per-species monthly histograms.

selected_subgroup_keys = list(subgroup_selector.value)
if not selected_subgroup_keys:
    print('Tier 3 skipped: select one subgroup in the Subgroups widget to run species detail.')
else:
    if len(selected_subgroup_keys) > 1:
        print('Multiple subgroups selected; using the first selection for Tier 3 detail.')

    # Tier 3 fetches one subgroup at a time to minimize API load
    tier3_key = selected_subgroup_keys[0]
    tier3_entry = get_seasonal_taxon_by_key(tier3_key)

    top_taxa_df, monthly_taxa_df = fetch_top_seasonal_taxa(
        tier3_entry,
        PIEDMONT_PLACE_IDS,
        session=session,
        top_n=TOP_N,
        min_obs_per_year=MIN_OBS_PER_YEAR,
        tier3_rank=TIER3_RANK,
        native=True,
        cache_path=CACHE_PATH,
        cache_tag=CACHE_TAG,
        cache_max_age_days=CACHE_MAX_AGE_DAYS,
        refresh_cache=REFRESH_CACHE,
    )

    print(f"Tier 3 species detail for: {tier3_entry['title_label']}")
    if not top_taxa_df.empty:
        preview = top_taxa_df.copy()
        preview['taxon_name'] = preview['taxon_name'].fillna(preview['display_label']).fillna(preview['taxon_id'].apply(lambda x: f'Taxon {int(x)}'))
        preview['common_name'] = preview['common_name'].fillna('')
        preview['taxon_rank'] = preview['taxon_rank'].fillna('(rank unavailable)')
        preview = preview[
            ['common_name', 'taxon_name', 'taxon_rank', 'taxon_id', 'total_obs']
        ].sort_values('total_obs', ascending=False)
        print(f"  Retained {len(top_taxa_df)} taxa (threshold: {MIN_OBS_PER_YEAR}/year)")
        print(f"  Missing common names: {int(top_taxa_df['common_name'].isna().sum())}")
        print(f"  Missing taxon ranks: {int(top_taxa_df['taxon_rank'].isna().sum())}")
        print()
        print(preview.to_string(index=False))
    else:
        print('  No taxa met threshold.')

    # -- Tier 3 chart: monthly species prevalence (within-taxon normalized) --
    month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

    if monthly_taxa_df.empty:
        print(f"No species passed threshold for {tier3_entry['title_label']}.")
    else:
        plot_df = monthly_taxa_df.copy()
        plot_df['month_name'] = plot_df['month'].apply(lambda m: month_order[m - 1])
        plot_df['month_name'] = pd.Categorical(plot_df['month_name'], categories=month_order, ordered=True)
        plot_df['label'] = (
            plot_df['display_label']
            .fillna(plot_df['common_name'])
            .fillna(plot_df['taxon_name'])
            .fillna(plot_df['taxon_id'].apply(lambda x: f'Taxon {int(x)}'))
        )

        analysis_mode = tier3_entry.get('analysis_mode', 'seasonality')
        if analysis_mode == 'phenology':
            title_prefix = 'Monthly phenology prevalence'
        else:
            title_prefix = 'Monthly seasonal prevalence'

        fig = px.line(
            plot_df.sort_values('month'),
            x='month_name',
            y='fraction',
            color='label',
            markers=True,
            category_orders={'month_name': month_order},
            labels={
                'month_name': '',
                'fraction': '% of annual observations',
                'label': 'Taxon',
            },
        )

        title = (
            f"{title_prefix} — top {TOP_N} native {tier3_entry['title_label']} "
            f"(within-taxon normalized, threshold {MIN_OBS_PER_YEAR}/year, rank={TIER3_RANK})"
        )
        fig = apply_plot_style(fig, title_text=title, height=620)
        fig.update_yaxes(tickformat='.1%')
        fig.show()

print('\nRun on: '+dt.datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

Tier 3 species detail for: Mammals
  Retained 12 taxa (threshold: 25/year)
  Missing common names: 0
  Missing taxon ranks: 0

                common_name             taxon_name taxon_rank  taxon_id  total_obs
          White-tailed Deer Odocoileus virginianus    species     42223      12213
      Eastern Gray Squirrel   Sciurus carolinensis    species     46017       8223
                    Red Fox          Vulpes vulpes    species     42069       3448
         Eastern Cottontail  Sylvilagus floridanus    species     43111       2634
           Eastern Chipmunk        Tamias striatus    species     46217       2177
             Common Raccoon          Procyon lotor    species     41663       1833
            American Beaver      Castor canadensis    species     43794       1225
                  Groundhog          Marmota monax    species     46095        958
                    Muskrat     Ondatra zibethicus    species     45763        745
                     Coyote          Canis 


Run on: 2026-04-04 15:41:23


In [5]:
## We can probably skip this cell except when appropriate to the selected taxon


# -- Life stage coverage (phenology-relevant taxa only) --
# Uses subgroup_hist from Tier 2 for any subgroup with life_stages defined.

phenology_entries = [e for e in subgroup_entries if e.get('life_stages')]

if not phenology_entries:
    print('No phenology life-stage data in the selected group.')
else:
    print('Life stage annotation coverage (phenology-relevant subgroups):')
    for entry in phenology_entries:
        fg = entry['focus_group']
        overall = subgroup_hist[
            (subgroup_hist['focus_group'] == fg)
            & (subgroup_hist['life_stage_bucket'] == 'overall')
            & (subgroup_hist['interval'] == 'week_of_year')
        ]['count'].sum()

        print()
        print(f"{entry['title_label']} — native observations (all-time): {int(overall):,}")
        for stage in entry['life_stages']:
            stage_count = subgroup_hist[
                (subgroup_hist['focus_group'] == fg)
                & (subgroup_hist['life_stage_bucket'] == stage['bucket'])
                & (subgroup_hist['interval'] == 'week_of_year')
            ]['count'].sum()
            pct = (stage_count / overall) if overall else float('nan')
            caution = ' (sparse, interpret cautiously)' if pct < 0.10 else ''
            print(f"  - {stage['bucket']}: {int(stage_count):,} ({pct:.1%}){caution}")

print('\nRun on: '+dt.datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

No phenology life-stage data in the selected group.

Run on: 2026-04-04 15:41:05
